In [19]:
import os
import json
import pandas as pd
from glob import glob
from bs4 import BeautifulSoup
from concurrent.futures import ThreadPoolExecutor

# Paths
jsonl_path = "/path/to/data/poets_gate.jsonl"
html_folder = "/path/to/data/raw/poets_gate/poets"
output_csv = "poets_gate_with_descriptions.csv"

def extract_poet_info_from_html_file(file_path):
    try:
        with open(file_path, "r", encoding="utf-8") as f:
            html = f.read()

        soup = BeautifulSoup(html, "lxml")

        profile_div = soup.find("div", id="profile")
        if not profile_div:
            return None

        lines = profile_div.get_text(separator="\n").splitlines()
        lines = [line.strip() for line in lines if line.strip()]
        if not lines:
            return None

        poet_name = lines[0]
        description = "\n".join(lines).strip()

        link_tag = soup.find("link", rel="canonical")
        url = link_tag["href"].strip() if link_tag and "href" in link_tag.attrs else ""

        return poet_name.strip(), {"description": description, "url": url}

    except Exception as e:
        print(f"Error processing {file_path}: {e}")
        return None

def build_description_lookup_parallel(html_dir, max_workers=8):
    html_files = glob(os.path.join(html_dir, "*.html"))
    lookup = {}

    with ThreadPoolExecutor(max_workers=max_workers) as executor:
        results = executor.map(extract_poet_info_from_html_file, html_files)

    for result in results:
        if result:
            poet_name, data = result
            lookup[poet_name] = data

    return lookup

def main():
    # Step 1: Load JSONL as DataFrame
    print("📄 Loading poets from JSONL...")
    with open(jsonl_path, "r", encoding="utf-8") as f:
        records = [json.loads(line) for line in f]
    df = pd.DataFrame(records)

    # Step 2: Build poet description lookup in parallel
    print("🚀 Extracting poet descriptions and URLs in parallel...")
    lookup = build_description_lookup_parallel(html_folder)

    # Step 3: Add description and URL columns to DataFrame
    print("🧩 Matching descriptions and URLs to poets...")
    df["poet_description"] = df["poet"].apply(lambda name: lookup.get(name.strip(), {}).get("description", ""))
    df["poet_description_url"] = df["poet"].apply(lambda name: lookup.get(name.strip(), {}).get("url", ""))

    # Step 4: Save to CSV
    print(f"💾 Saving to CSV: {output_csv}")
    df.to_csv(output_csv, index=False, encoding="utf-8")

    print("✅ Done.")


main()


📄 Loading poets from JSONL...


🚀 Extracting poet descriptions and URLs in parallel...
🧩 Matching descriptions and URLs to poets...
💾 Saving to CSV: poets_gate_with_descriptions.csv
✅ Done.


In [22]:
import pandas as pd
poets_gate_data  = pd.read_csv('poets_gate_with_descriptions.csv')

poets_gate_data["poet_page_url"] = poets_gate_data.pop("poet_description_url")
poets_gate_data.to_csv('poets_gate_with_descriptions.csv', index=False, encoding="utf-8")

In [24]:
unknown_title_count = poets_gate_data[poets_gate_data['title'] == 'Unknown'].shape[0]
print(f"Number of rows with unknown title: {unknown_title_count}")

Number of rows with unknown title: 15444
